# Run MoneyPrinterTurbo on Google Colab\n\nThis notebook installs the active `feat/scene-composition-engine` branch in an isolated Python 3.11 environment and launches its WebUI. Generated task folders and task-state records are stored in Google Drive so completed outputs and task history survive Colab runtime resets. A runtime reset still stops the Streamlit process; in-progress generation tasks are marked interrupted on the next startup rather than silently remaining stuck or being falsely resumed.\n

## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/methaneex-oss/MoneyPrinterTurbo.git"
REPO_BRANCH = "feat/scene-composition-engine"

# Always install the active feature branch; main does not yet contain this work.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH],
        check=True,
    )
    local_branch_exists = subprocess.run(
        ["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet",
         f"refs/heads/{REPO_BRANCH}"],
        check=False,
    ).returncode == 0
    if local_branch_exists:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", "-b", REPO_BRANCH,
             f"origin/{REPO_BRANCH}"],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")

In [ ]:
# Keep generated task folders in Google Drive so Colab runtime resets do not erase finished videos.
from google.colab import drive
from pathlib import Path
import filecmp
import shutil

drive.mount('/content/drive')

REPO_DIR = Path('/content/MoneyPrinterTurbo')
TASKS_DIR = REPO_DIR / 'storage' / 'tasks'
DRIVE_TASKS_DIR = Path('/content/drive/MyDrive/MoneyPrinterTurbo/tasks')
DRIVE_TASKS_DIR.mkdir(parents=True, exist_ok=True)
TASKS_DIR.parent.mkdir(parents=True, exist_ok=True)

def merge_runtime_item(source: Path, destination: Path) -> None:
    """Merge without discarding either side when an existing file differs."""
    if source.is_dir():
        if destination.exists() and not destination.is_dir():
            raise RuntimeError(f'Cannot merge directory over a file: {destination}')
        destination.mkdir(parents=True, exist_ok=True)
        for child in source.iterdir():
            merge_runtime_item(child, destination / child.name)
        return

    if not source.is_file():
        raise RuntimeError(f'Unsupported task artifact type: {source}')

    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if not destination.is_file():
            raise RuntimeError(f'Cannot merge file over a directory: {destination}')
        if filecmp.cmp(source, destination, shallow=False):
            return

        # Keep the existing Drive version before the current runtime version
        # replaces it. This also makes a partially completed prior migration safe.
        backup = destination.with_name(destination.name + '.drive-backup')
        suffix = 1
        while backup.exists():
            backup = destination.with_name(
                destination.name + f'.drive-backup-{suffix}'
            )
            suffix += 1
        shutil.copy2(destination, backup)

    shutil.copy2(source, destination)


# Migrate any task outputs already in this runtime before linking the folders.
if TASKS_DIR.is_symlink():
    current_target = TASKS_DIR.resolve()
    if current_target != DRIVE_TASKS_DIR.resolve():
        raise RuntimeError(f'Unexpected tasks symlink target: {current_target}')
elif TASKS_DIR.exists():
    for item in TASKS_DIR.iterdir():
        merge_runtime_item(item, DRIVE_TASKS_DIR / item.name)

    # Verify every local file against its Drive copy before deleting anything.
    for source_file in TASKS_DIR.rglob('*'):
        if source_file.is_file():
            destination = DRIVE_TASKS_DIR / source_file.relative_to(TASKS_DIR)
            if not destination.is_file() or not filecmp.cmp(
                source_file, destination, shallow=False
            ):
                raise RuntimeError(
                    f'Task migration verification failed: {source_file}'
                )

    shutil.rmtree(TASKS_DIR)
    TASKS_DIR.symlink_to(DRIVE_TASKS_DIR, target_is_directory=True)
else:
    TASKS_DIR.symlink_to(DRIVE_TASKS_DIR, target_is_directory=True)

print(f'Generated videos and task files will be saved in: {DRIVE_TASKS_DIR}')

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/MoneyPrinterTurbo')
CONFIG_PATH = REPO_DIR / 'config.toml'

# Create the standard config only on first run. Existing API keys and preferences are preserved.
if not CONFIG_PATH.exists():
    import shutil
    shutil.copy2(REPO_DIR / 'config.example.toml', CONFIG_PATH)

# Apply the documentary preset once. Later notebook runs preserve any settings you change in the WebUI.
preset_script = r'''
from pathlib import Path
import toml

config_path = Path('config.toml')
config = toml.load(config_path)
app = config.setdefault('app', {})
ui = config.setdefault('ui', {})
preset_version = int(app.get('visual_storytelling_preset_version', 0) or 0)

if preset_version < 1:
    app['match_materials_to_script'] = True
    app['visual_storytelling_preset_version'] = 1
    ui.update({
        'video_concat_mode': 'sequential',
        'video_transition_mode': 'FadeIn',
        'video_aspect_pixabay': '9:16',
        'video_fit_mode': 'cover',
        'video_clip_duration': 4,
        'video_clip_speed': 1.0,
        'bgm_volume': 0.1,
        'subtitle_position': 'two_thirds_bottom',
        'subtitle_display_mode': 'sentence',
        'subtitle_background_enabled': True,
        'subtitle_background_color': '#000000',
        'rounded_subtitle_background': True,
    })
    with config_path.open('w', encoding='utf-8') as handle:
        toml.dump(config, handle)
    print('Documentary editing preset applied. Existing API keys were preserved.')
else:
    print('Documentary editing preset already applied; current WebUI preferences were left unchanged.')

# Persist task status metadata as well as task folders across Colab runtime resets.
# This path is on the already-mounted Google Drive and is enabled by FileState.
app['task_state_dir'] = '/content/drive/MyDrive/MoneyPrinterTurbo/task-state'
app['enable_redis'] = False
app['visual_storytelling_preset_version'] = max(preset_version, 2)
with config_path.open('w', encoding='utf-8') as handle:
    toml.dump(config, handle)
print('Persistent task state is configured on Google Drive.')
'''

subprocess.run(['uv', 'run', 'python', '-c', preset_script], cwd=REPO_DIR, check=True)


## 2. Enable free local AI scene images (GPU runtimes only)

This step selects a local image model from the GPU actually assigned by Colab. With at least 14 GiB VRAM it uses RealVisXL SDXL Lightning; with 8–14 GiB it uses Stability AI SD-Turbo, a one-step lower-VRAM model. Below 8 GiB, it safely falls back to the configured Pixabay source.

Model weights are downloaded on first warm-up and cached in the runtime. The local models generate still images; MoneyPrinterTurbo then adds camera motion and its scene-graphic system. This is not AI text-to-video. No paid image API is called.

In [ ]:
from pathlib import Path
import json
import subprocess
import time
from urllib.error import URLError
from urllib.request import Request, urlopen

REPO_DIR = Path("/content/MoneyPrinterTurbo")
IMAGE_SERVER_DIR = REPO_DIR / "docs" / "local-image-server"
IMAGE_ENV = REPO_DIR / ".venv-image"
IMAGE_PYTHON = IMAGE_ENV / "bin" / "python"
IMAGE_LOG_PATH = Path("/content/mpt-image-server.log")
IMAGE_PORT = 7861

def stop_image_server():
    process = globals().get("image_server_proc")
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=15)
    log_handle = globals().get("image_server_log")
    if log_handle is not None and not log_handle.closed:
        log_handle.close()

def set_video_source(source, base_url="", model="", size=""):
    script = f"""
import toml
from pathlib import Path
path = Path("config.toml")
cfg = toml.load(path) if path.exists() else {{}}
app = cfg.setdefault("app", {{}})
app["video_source"] = {source!r}
if {source!r} == "openai_image":
    app["openai_image_base_url"] = {base_url!r}
    app["openai_image_model"] = {model!r}
    app["openai_image_size"] = {size!r}
    app["openai_image_api_keys"] = []
with path.open("w", encoding="utf-8") as handle:
    toml.dump(cfg, handle)
"""
    subprocess.run(["uv", "run", "python", "-c", script], cwd=REPO_DIR, check=True)

def use_pixabay(reason):
    stop_image_server()
    set_video_source("pixabay")
    print(f"Local AI images unavailable ({reason}). Using the configured Pixabay source.")

stop_image_server()

try:
    import torch

    gpu_ready = torch.cuda.is_available()
    gpu_memory_gib = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        if gpu_ready else 0
    )
    print(f"CUDA available: {gpu_ready}; GPU memory: {gpu_memory_gib:.1f} GiB")

    if not gpu_ready or gpu_memory_gib < 8:
        use_pixabay("Colab did not provide a CUDA GPU with at least 8 GiB VRAM")
    else:
        # Prefer the higher-quality SDXL Lightning path when memory permits.
        # SD-Turbo is the lower-VRAM, one-step fallback for 8-14 GiB GPUs.
        image_backend = "sdxl" if gpu_memory_gib >= 14 else "sd-turbo"
        image_model = (
            "SG161222/RealVisXL_V5.0_Lightning"
            if image_backend == "sdxl"
            else "stabilityai/sd-turbo"
        )
        image_size = "832x1216" if image_backend == "sdxl" else "512x768"
        print(f"Selected local image model: {image_backend} ({image_model})")
        subprocess.run(
            ["uv", "venv", "--python", "3.11", "--system-site-packages", str(IMAGE_ENV)],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            [
                "uv", "pip", "install", "--python", str(IMAGE_PYTHON),
                "-r", str(IMAGE_SERVER_DIR / "requirements.txt"),
            ],
            cwd=REPO_DIR,
            check=True,
        )

        image_server_log = IMAGE_LOG_PATH.open("w", encoding="utf-8")
        image_server_env = os.environ.copy()
        image_server_env["IMAGE_BACKEND"] = image_backend
        image_server_env["IMAGE_MODEL"] = image_model
        image_server_proc = subprocess.Popen(
            [str(IMAGE_PYTHON), "server.py"],
            cwd=IMAGE_SERVER_DIR,
            env=image_server_env,
            stdout=image_server_log,
            stderr=subprocess.STDOUT,
            text=True,
        )

        deadline = time.time() + 90
        image_server_ready = False
        while time.time() < deadline:
            if image_server_proc.poll() is not None:
                break
            try:
                with urlopen(f"http://127.0.0.1:{IMAGE_PORT}/health", timeout=2) as response:
                    image_server_ready = response.status == 200
            except (URLError, TimeoutError):
                pass
            if image_server_ready:
                break
            time.sleep(2)

        if not image_server_ready:
            raise RuntimeError("local image server did not become healthy")

        # Load the weights now, before the WebUI starts. This prevents the
        # first generation request from timing out while downloading 6.6 GB.
        warmup_request = Request(
            f"http://127.0.0.1:{IMAGE_PORT}/warmup",
            data=b"",
            method="POST",
        )
        with urlopen(warmup_request, timeout=900) as response:
            warmup_result = json.loads(response.read().decode("utf-8"))
        if (
            not warmup_result.get("ok")
            or not warmup_result.get("on_gpu")
            or warmup_result.get("backend") != image_backend
            or warmup_result.get("model") != image_model
        ):
            raise RuntimeError(f"local image model warmup was not successful: {warmup_result}")

        set_video_source(
            "openai_image",
            f"http://127.0.0.1:{IMAGE_PORT}/v1",
            image_model,
            image_size,
        )
        print(
            f"Local {image_backend} image generation is ready. "
            "No image API key or paid provider is used."
        )

except Exception as error:
    try:
        use_pixabay(f"{type(error).__name__}: {error}")
    except Exception as config_error:
        print(f"Could not apply the Pixabay fallback: {config_error}")
    print("The WebUI can still launch using the configured stock-video workflow.");


## 3. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.

In [ ]:
from getpass import getpass

from pyngrok import ngrok

# Close tunnels from earlier runs before configuring a new one.
ngrok.kill()

# Add NGROK_AUTHTOKEN once in Colab's Secrets panel (key icon).
# The prompt remains as a fallback if the secret is not available.
ngrok_token = None
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    pass

if not ngrok_token:
    ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

## 4. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys. To use OpenRouter free LLMs, select **OpenRouter**, enter your OpenRouter API key, and keep the default `minimax/minimax-m3:free` model or choose another model from https://openrouter.ai/models?variant=free.

In [ ]:
import time
from pathlib import Path
import subprocess
from urllib.error import URLError
from urllib.request import urlopen

from pyngrok import ngrok

# Sanity checks before starting
REPO_DIR = Path("/content/MoneyPrinterTurbo")
if not REPO_DIR.is_dir():
    raise RuntimeError(f"Repo directory does not exist: {REPO_DIR}")
if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(f"webui/Main.py not found in {REPO_DIR}")

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

try:
    ngrok.kill()
except Exception:
    pass

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}")
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")